# 04 Feature Engineering

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]


In [4]:
bucket = "ads508-team-4-private"
input_path = f"s3://{bucket}/processed/combined_attrition_linkedin_data.csv"

df = pd.read_csv(input_path)
df.head()

,Age,Attrition,Department,DistanceFromHome,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,...,RelationshipSatisfaction,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,JobRoleMapped,external_avg_salary,external_job_demand,salary_gap
0,41,Yes,Sales,1,1,2,Female,94,3,2,...,1,0,1,6,4,0,Sales Executive,91683.754386,139,85690.754386
1,49,No,Research & Development,8,2,3,Male,61,2,2,...,4,3,3,10,7,1,Research Scientist,89817.885930,119,84687.885930
2,37,Yes,Research & Development,2,4,4,Male,92,2,1,...,2,3,3,0,0,0,Laboratory Technician,17372.834231,33,15282.834231
3,33,No,Research & Development,3,5,4,Female,56,3,1,...,3,3,3,8,7,3,Research Scientist,89817.885930,119,86908.885930
4,27,No,Research & Development,2,7,1,Male,40,3,1,...,4,3,3,2,2,2,Laboratory Technician,17372.834231,33,13904.834231


In [5]:
df = df.drop(columns=["EmployeeNumber"], errors="ignore")

In [6]:
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})
df["Attrition"].value_counts(normalize=True)

Attrition
0    0.805231
1    0.194769
Name: proportion, dtype: float64

In [7]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = df.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Categorical columns: ['Department', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'JobRoleMapped']
Numeric columns: ['Age', 'Attrition', 'DistanceFromHome', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PerformanceRating', 'RelationshipSatisfaction', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'external_avg_salary', 'external_job_demand', 'salary_gap']


In [8]:
# create income buckets
df["IncomeCategory"] = pd.qcut(
    df["MonthlyIncome"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [9]:
# create tenure group 
df["TenureCategory"] = pd.cut(
    df["YearsAtCompany"],
    bins=[-1, 2, 5, 10, 100],
    labels=["0-2 years", "3-5 years", "6-10 years", "10+ years"]
)

In [10]:
# create promotion flag
df["NoRecentPromotion"] = np.where(df["YearsSinceLastPromotion"] >= 3, 1, 0)

In [11]:
df[["MonthlyIncome", "IncomeCategory", "YearsAtCompany", "TenureCategory", "YearsSinceLastPromotion", "NoRecentPromotion"]].head()

,MonthlyIncome,IncomeCategory,YearsAtCompany,TenureCategory,YearsSinceLastPromotion,NoRecentPromotion
0,5993,Low,6,6-10 years,0,0
1,5130,Low,10,6-10 years,1,0
2,2090,Low,0,0-2 years,0,0
3,2909,Low,8,6-10 years,3,1
4,3468,Low,2,0-2 years,2,0


In [12]:
#create stagnation score, and age buckets and satsifaction index
df["StagnationScore"] = (
    df["YearsInCurrentRole"] + df["YearsSinceLastPromotion"]
)

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[18, 25, 35, 45, 55, 70],
    labels=["18-25", "26-35", "36-45", "46-55", "55+"],
    include_lowest=True
).astype(str)

sat_cols = [c for c in ["JobSatisfaction", "EnvironmentSatisfaction", "RelationshipSatisfaction"] if c in df.columns]
df["SatisfactionIndex"] = df[sat_cols].mean(axis=1).round(2)


In [13]:
# Count missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing

Age                         0
Attrition                   0
Department                  0
DistanceFromHome            0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
NumCompaniesWorked          0
OverTime                    0
PerformanceRating           0
RelationshipSatisfaction    0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
JobRoleMapped               0
external_avg_salary         0
external_job_demand         0
salary_gap                  0
IncomeCategory              0
TenureCategory              0
NoRecentPromotion           0
StagnationScore             0
AgeGroup                    0
SatisfactionIndex           0
dtype: int64

In [15]:
#one hot encode variable
df_model = pd.get_dummies(df, drop_first=True)

bool_cols = df_model.select_dtypes(include="bool").columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print("Encoded dataset shape:", df_model.shape)

Encoded dataset shape: (11470, 58)


In [16]:
bucket = "ads508-team-4-private"

output_path = f"s3://{bucket}/processed/feature_eng_combined_attrition_linkedin_data.csv"

df_model.to_csv(output_path, index=False)

In [17]:
#seperate test variable
selected_features = [
    "Age",
    "MaritalStatus_Single",
    "OverTime_Yes",
    "JobInvolvement",
    "WorkLifeBalance",
    "StagnationScore",
    "NoRecentPromotion",
    "SatisfactionIndex",
    "salary_gap",
    "external_job_demand",
    "TenureCategory_3-5 years",
    "TenureCategory_6-10 years",
    "TenureCategory_10+ years",
    "IncomeCategory_Medium",
    "IncomeCategory_High",
]

# Keep only existing columns (safe)
selected_features = [c for c in selected_features if c in df_model.columns]

X = df_model[selected_features]
y = df_model["Attrition"]

print("Final feature count:", len(selected_features))

Final feature count: 27


In [18]:
#Split into train (80%) and temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

# Split temp into validation (10%) and test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,  
    random_state=42,
    stratify=y_temp
)

In [19]:
smote = SMOTE(random_state=1)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [20]:
print("Train:", y_train_resampled.value_counts(normalize=True))
print("Validation:", y_val.value_counts(normalize=True))
print("Test:", y_test.value_counts(normalize=True))

Train: Attrition
1    0.5
0    0.5
Name: proportion, dtype: float64
Validation: Attrition
0    0.804708
1    0.195292
Name: proportion, dtype: float64
Test: Attrition
0    0.80558
1    0.19442
Name: proportion, dtype: float64


### Save data

In [21]:
bucket = "ads508-team-4-private"

# Train (Original)
train_original_df = X_train.copy()
train_original_df["Attrition"] = y_train

# Train (SMOTE)
train_smote_df = pd.DataFrame(X_train_resampled, columns=X_train.columns)
train_smote_df["Attrition"] = y_train_resampled

# Validation
val_df = X_val.copy()
val_df["Attrition"] = y_val

# Test
test_df = X_test.copy()
test_df["Attrition"] = y_test

# Paths
train_smote_path = f"s3://{bucket}/processed/train_attrition_smote.csv"
val_path = f"s3://{bucket}/processed/validation_attrition_data.csv"
test_path = f"s3://{bucket}/processed/test_attrition_data.csv"
train_original_path = f"s3://{bucket}/processed/train_attrition_original.csv"
train_original_df.to_csv(train_original_path, index=False)

# Save to S3
train_smote_df.to_csv(train_smote_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(train_smote_path)
print(val_path)
print(test_path)
print(train_original_path)

s3://ads508-team-4-private/processed/train_attrition_smote.csv
s3://ads508-team-4-private/processed/validation_attrition_data.csv
s3://ads508-team-4-private/processed/test_attrition_data.csv
s3://ads508-team-4-private/processed/train_attrition_original.csv


In [22]:
# download from S3
!aws s3 cp s3://ads508-team-4-private/processed/train_attrition_smote.csv ./data/processed/
!aws s3 cp s3://ads508-team-4-private/processed/validation_attrition_data.csv ./data/processed/
!aws s3 cp s3://ads508-team-4-private/processed/test_attrition_data.csv ./data/processed/
!aws s3 cp s3://ads508-team-4-private/processed/train_attrition_original.csv ./data/processed/

download: s3://ads508-team-4-private/processed/train_attrition_smote.csv to data/processed/train_attrition_smote.csv
download: s3://ads508-team-4-private/processed/validation_attrition_data.csv to data/processed/validation_attrition_data.csv
download: s3://ads508-team-4-private/processed/test_attrition_data.csv to data/processed/test_attrition_data.csv
download: s3://ads508-team-4-private/processed/train_attrition_original.csv to data/processed/train_attrition_original.csv
